# Análisis del Motor de Búsqueda Semántica

Este notebook contiene el análisis de las búsquedas semánticas sobre la colección de artículos de tecnología usando embeddings de OpenAI y ChromaDB.

## Contenidos:
1. **Importaciones y Setup** - Configuración inicial
2. **Tabla Comparativa** - Resultados para cada query
3. **Visualización de Scores** - Gráficos de similitud
4. **Mapa de Calor** - Matriz de similitud entre artículos
5. **Análisis Detallado** - Todos los resultados por query

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from buscar import buscar
from articulos import articulos

In [ ]:
queries = [
    "¿cómo hacer una API en Python?",
    "diferencias entre frameworks de frontend",
    "cómo funciona la autenticación en aplicaciones web",
    "herramientas para trabajar con modelos de lenguaje"
]

filas = []
for q in queries:
    resultados = buscar(q, n_resultados=3)
    mejor = resultados[0]
    filas.append({
        "query": q,
        "mejor_resultado": mejor["titulo"],
        "score": mejor["score"]
    })

df = pd.DataFrame(filas)
df

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x="score", y="query")
plt.title("Mejor score por query")
plt.tight_layout()
plt.show()

In [ ]:
# BONUS: Mapa de calor de similitud entre artículos
import numpy as np
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
client_matrix = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "text-embedding-3-small"

# Crear embeddings para todos los artículos
textos = [f"{a['titulo']}. {a['contenido']}" for a in articulos]
respuesta = client_matrix.embeddings.create(model=MODEL, input=textos)
embeddings = np.array([item.embedding for item in respuesta.data])

# Calcular matriz de similitud (coseno)
from sklearn.metrics.pairwise import cosine_similarity
matriz_similitud = cosine_similarity(embeddings)

# Visualizar como heatmap
plt.figure(figsize=(10, 8))
titulos_cortos = [a['titulo'][:20] + '...' if len(a['titulo']) > 20 else a['titulo'] for a in articulos]
sns.heatmap(matriz_similitud, 
            xticklabels=titulos_cortos, 
            yticklabels=titulos_cortos,
            cmap='coolwarm', 
            annot=True, 
            fmt='.2f',
            cbar_kws={'label': 'Similitud (Coseno)'})
plt.title('Mapa de Calor: Similitud entre Artículos', fontsize=14, fontweight='bold')
plt.xlabel('Artículos', fontsize=11)
plt.ylabel('Artículos', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico: Top 5 scores de similitud
import numpy as np

plt.figure(figsize=(14, 6))

# Subgráfico 1: Barras de scores
ax1 = plt.subplot(1, 2, 1)
df_sorted = df.sort_values('score', ascending=True)
sns.barplot(data=df_sorted, y='query', x='score', palette='viridis', ax=ax1)
ax1.set_title('Score de similitud por Query', fontsize=12, fontweight='bold')
ax1.set_xlabel('Score de similitud', fontsize=11)

# Subgráfico 2: Comparación query vs score
ax2 = plt.subplot(1, 2, 2)
colors = plt.cm.RdYlGn(df['score'] / df['score'].max())
ax2.barh(range(len(df)), df['score'], color=colors)
ax2.set_yticks(range(len(df)))
ax2.set_yticklabels([q[:40] + '...' if len(q) > 40 else q for q in df['query']], fontsize=10)
ax2.set_xlabel('Score de similitud', fontsize=11)
ax2.set_title('Comparativa de Scores', fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Todos los resultados por query
print("\nDETALLES DE TODAS LAS BÚSQUEDAS:")
for q in queries:
    print(f"\n📌 Query: {q}")
    resultados = buscar(q, n_resultados=3)
    for i, r in enumerate(resultados, 1):
        print(f"   {i}. [{r['score']:.4f}] {r['titulo']}")

In [ ]:
# Tabla comparativa detallada
print("\n" + "="*80)
print("TABLA COMPARATIVA: Query → Mejor Resultado → Score")
print("="*80)
df_display = df.copy()
df_display['score'] = df_display['score'].round(4)
print(df_display.to_string(index=False))
print("="*80)